In [1]:
import torch
import os
# Set CUDA memory management before importing torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

import torch.optim as optim
from tqdm import tqdm
import data as Data
import models as Model
import torch.nn as nn
import argparse
import logging
import core.logger as Logger
from core.utils import *
import numpy as np
from misc.metric_tools import ConfuseMatrixMeter
from models.loss import *
from collections import OrderedDict
import core.metrics as Metrics
from misc.torchutils import get_scheduler, save_network
import wandb
import matplotlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datetime import datetime
from itertools import islice

In [2]:
import argparse, os
from datetime import datetime

# ---------- CHANGE 1: robust seed helper ----------
import random, numpy as np, torch
def set_seed(seed: int | None):
    if seed is None:
        print("[seed] None → skipping reproducibility setup")
        return
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[seed] Set global seed = {seed}")

# ---------- args (unchanged structure; simulated CLI for notebooks) ----------
parser = argparse.ArgumentParser()
parser.add_argument('--config', type=str, default='/workspace/BuildingCD_mamba_based/config/second_cdmamba/second_cdmamba.json')
parser.add_argument('--phase', type=str, default='train', choices=['train', 'test'])
parser.add_argument('--model', type=str, default='')
parser.add_argument('--dataset', type=str, default='')
parser.add_argument('--tag', type=str, default='')
parser.add_argument('--seed', type=int, default=None)
parser.add_argument('--max_train_batches', type=int, default=0)
parser.add_argument('--max_val_batches', type=int, default=0)
parser.add_argument('--max_test_batches', type=int, default=0)
parser.add_argument('--change_threshold', type=float, default=0.5)  # CHANGE 3: default 0.5

notebook_args = [
    "--config", "/workspace/BuildingCD_mamba_based/config/second_cdmamba/second_cdmamba_modified.json",
    "--phase", "train",
    "--change_threshold", "0.5",           # keep at 0.5 for debug; eval-only
    "--seed", "42",
    "--tag", "debug",
    "--max_train_batches", "1",
    "--max_val_batches", "1",
    "--max_test_batches", "1"
]
args = parser.parse_args(notebook_args)

# Your logger parse flow
opt = Logger.parse(args)
opt = Logger.dict_to_nonedict(opt)

# ---------- CHANGE 5: richer experiment folder naming ----------
exp_timestamp = datetime.now().strftime('%m%d_%H')
exp_name = opt.get('name', 'experiment')
dataset_suffix = getattr(args, 'dataset', None) or ''
tag_suffix = getattr(args, 'tag', None) or ''
seed_suffix = f"seed{args.seed}" if getattr(args, "seed", None) is not None else ""
parts = [p for p in [dataset_suffix, tag_suffix, seed_suffix] if p]
suffix = "_".join(parts)
exp_folder = f"{suffix}_{exp_timestamp}" if suffix else f"{exp_timestamp}"

# ---------- CHANGE 4: guard and create stamped dirs ----------
if 'path_cd' in opt and isinstance(opt['path_cd'], dict):
    for k in ['log', 'result', 'checkpoint']:
        if k in opt['path_cd'] and isinstance(opt['path_cd'][k], str):
            base_dir = opt['path_cd'][k]
            stamped = os.path.join(base_dir, exp_folder)
            opt['path_cd'][k] = stamped
            os.makedirs(stamped, exist_ok=True)
    opt['path_cd']['exp_folder'] = exp_folder
else:
    print("[warn] opt['path_cd'] not found; skipping folder stamping")

# ---------- CHANGE 1 (call): seed ASAP ----------
set_seed(args.seed if args.seed is not None else 42)

# ---------- CHANGE 6: print resolved header ----------
print("[run] name:", opt.get('name', 'experiment'))
print("[run] phase:", args.phase, "| tag:", args.tag, "| seed:", args.seed)
print("[run] change_threshold (eval only):", args.change_threshold)
print("[run] exp_folder:", opt.get('path_cd', {}).get('exp_folder', '<none>'))


[seed] Set global seed = 42
[run] name: SECOND-CDMamba
[run] phase: train | tag: debug | seed: 42
[run] change_threshold (eval only): 0.5
[run] exp_folder: debug_seed42_0825_21


In [3]:
#logging
torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = False

Logger.setup_logger(logger_name=None, root=opt['path_cd']['log'], phase='train',
                    level=logging.INFO, screen=True)
Logger.setup_logger(logger_name='val', root=opt['path_cd']['log'], phase='val',
                    level=logging.INFO)
Logger.setup_logger(logger_name='test', root=opt['path_cd']['log'], phase='test',
                    level=logging.INFO)
logger = logging.getLogger('base')
logger.info(Logger.dict2str(opt))

# Set device with comprehensive debugging
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Using device: {device}')

WANDB_ENABLED = False
use_wandb = bool(opt.get('wandb') and opt['wandb'].get('project'))

if use_wandb:
    run_name = exp_folder
    try:
        wandb.init(project=opt['wandb']['project'], name=run_name)
        WANDB_ENABLED = getattr(wandb, "run", None) is not None
    except Exception as e:
        logger.warning(f"W&B init failed: {e}. Disabling.")
        try:
            wandb.init(mode="disabled")
        except Exception:
            pass
        WANDB_ENABLED = False
else:
    try:
        wandb.init(mode="disabled")
    except Exception:
        pass
    WANDB_ENABLED = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(args.seed if args.seed is not None else 42)

#dataset
for phase, dataset_opt in opt['datasets'].items(): #train train{}
    #print(" phase is {}, dataopt is {}".format(phase, dataset_opt))
    if phase == 'train' and args.phase != 'test':
        print("Creat [train] change-detection dataloader")
        train_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        train_loader = Data.create_cd_dataloader(train_set, dataset_opt, phase, seed_worker, g)
        opt['len_train_dataloader'] = len(train_loader)

    elif phase == 'val' and args.phase != 'test':
        print("Creat [val] change-detection dataloader")
        val_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        val_loader = Data.create_cd_dataloader(val_set, dataset_opt, phase, seed_worker, g)
        opt['len_val_dataloader'] = len(val_loader)

    # elif phase == 'test' and args.phase == 'test':
    elif phase == 'test':
        print("Creat [test] change-detection dataloader")
        test_set = Data.create_scd_dataset(dataset_opt=dataset_opt, phase=phase)
        test_loader = Data.create_cd_dataloader(test_set, dataset_opt, phase, seed_worker, g)
        opt['len_test_dataloader'] = len(test_loader)

logger.info('Initial Dataset Finished')

#Create cd model
cd_model = Model.create_CD_model(opt)

# Initialize model weights to prevent NaN loss - more conservative
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.xavier_normal_(m.weight, gain=0.1)  # Very small gain
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0, 0.001)  # Very small std
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

cd_model.apply(init_weights)
cd_model.to(device)
logger.info(f'CD Model moved to device: {device}')

# Verify model is actually on GPU
if torch.cuda.is_available():
    model_device = next(cd_model.parameters()).device
    logger.info(f'Model parameters are on device: {model_device}')
    if model_device.type != 'cuda':
        logger.error('WARNING: Model parameters are NOT on GPU!')
    else:
        logger.info('✓ Model successfully moved to GPU')

# Enable gradient checkpointing if available to save memory
if hasattr(cd_model, 'gradient_checkpointing_enable'):
    cd_model.gradient_checkpointing_enable()

num_classes = opt['model']['n_classes']
logger.info(f"Number of classes for loss function: {num_classes}")

#Create criterion (segmentation losses use semantic num_classes; change head will use 2)
if opt['model']['loss'] == 'ce_dice':
    loss_fun = CEDiceLoss(num_classes=num_classes)
    loss_fun_change = CEDiceLoss(num_classes=2)
elif opt['model']['loss'] == 'ce':
    # CrossEntropy can be used as a function or nn.Module. Using function for now.
    loss_fun = cross_entropy_loss_fn
    loss_fun_change = cross_entropy_loss_fn
elif opt['model']['loss'] == 'dice':
    loss_fun = DiceOnlyLoss(num_classes=num_classes)
    loss_fun_change = DiceOnlyLoss(num_classes=2)
elif opt['model']['loss'] == 'extended_triplet':
    # Extended multi-task loss: seg(t1)+seg(t2)+change + cross-time consistency + coupling
    base_seg = CEDiceLoss(num_classes=num_classes)
    cfg = opt['model'].get('extended_triplet', {})
    loss_fun = TripletChangeSegLoss(
        seg_loss_fn=base_seg,
        lambda_seg=cfg.get('lambda_seg', 1.0),
        lambda_cd=cfg.get('lambda_cd', 1.0),
        lambda_unch=cfg.get('lambda_unch', 0.2),
        lambda_ch=cfg.get('lambda_ch', 0.2),
        lambda_cpl=cfg.get('lambda_cpl', 0.5),
        T=cfg.get('T', 4.0),
        margin=cfg.get('margin', 0.3)
    )
else:
    raise ValueError(f"Unsupported loss function type: {opt['model']['loss']}")

# If losses are nn.Module, move them to the device
if isinstance(loss_fun, nn.Module):
    loss_fun.to(device)
if 'loss_fun_change' in locals() and isinstance(loss_fun_change, nn.Module):
    loss_fun_change.to(device)
# Fallback: if loss_fun_change wasn't defined (e.g., for unsupported options), reuse loss_fun
if 'loss_fun_change' not in locals():
    loss_fun_change = loss_fun

#Create optimizer
if opt['train']["optimizer"]["type"] == 'adam':
    beta1 = opt['train']["optimizer"].get("beta1", 0.9)  # fallback default
    beta2 = opt['train']["optimizer"].get("beta2", 0.999)
    optimizer = optim.Adam(
        cd_model.parameters(),
        lr=opt['train']["optimizer"]["lr"],
        betas=(beta1, beta2)
    )
elif opt['train']["optimizer"]["type"] == 'adamw':
    optimizer = optim.AdamW(cd_model.parameters(), lr=opt['train']["optimizer"]["lr"])
elif opt['train']["optimizer"]["type"] == 'sgd':
    optimizer = optim.SGD(cd_model.parameters(), lr=opt['train']["optimizer"]["lr"],
                        momentum=0.9, weight_decay=5e-4)

# Initialize mixed precision scaler
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

metric = ConfuseMatrixMeter(n_class=2)  # For binary change detection
metric_seg = ConfuseMatrixMeter(n_class=opt['model']['n_classes'])  # For 6-class segmentation
log_dict = OrderedDict()

if torch.cuda.is_available():
    try:
        torch.cuda.set_per_process_memory_fraction(0.8)
    except Exception as e:
        logger.warning(f"set_per_process_memory_fraction failed: {e}")


25-08-25 21:18:56.643 - INFO:   name: SECOND-CDMamba
  phase: train
  gpu_ids: [0]
  wandb:[
    project: mamba-cd-test
  ]
  path_cd:[
    log: /root/home/pvc/Building_changedetection_job/experiments/logs/debug_seed42_0825_21
    result: /root/home/pvc/Building_changedetection_job/experiments/results/debug_seed42_0825_21
    checkpoint: /root/home/pvc/Building_changedetection_job/experiments/checkpoint/debug_seed42_0825_21
    resume_state: None
    experiments_root: experiments/SECOND-CDMamba_250825_211856
    exp_folder: debug_seed42_0825_21
  ]
  datasets:[
    train:[
      name: SECOND-CD-256
      datasetroot: /root/home/pvc/SECOND/train
      resolution: 512
      num_workers: 4
      batch_size: 2
      use_shuffle: True
      data_len: -1
    ]
    val:[
      name: SECOND-CD-256
      datasetroot: /root/home/pvc/SECOND/val
      resolution: 512
      num_workers: 4
      batch_size: 2
      use_shuffle: False
      data_len: -1
    ]
    test:[
      name: SECOND-CD-256
    

/root/home/pvc/Building_changedetection_job/experiments/logs/debug_seed42_0825_21/train.log
/root/home/pvc/Building_changedetection_job/experiments/logs/debug_seed42_0825_21/val.log
/root/home/pvc/Building_changedetection_job/experiments/logs/debug_seed42_0825_21/test.log


wandb: Currently logged in as: saraashojaeii (saraashojaeii-university-of-missouri-system) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


25-08-25 21:18:59.515 - INFO: Dataset [SCDDataset - SECOND-CD-256 - train] is created
25-08-25 21:18:59.517 - INFO: Dataset [SCDDataset - SECOND-CD-256 - val] is created
25-08-25 21:18:59.518 - INFO: Dataset [SCDDataset - SECOND-CD-256 - test] is created
25-08-25 21:18:59.519 - INFO: Initial Dataset Finished


Creat [train] change-detection dataloader
/root/home/pvc/SECOND/train
Creat [val] change-detection dataloader
/root/home/pvc/SECOND/val
Creat [test] change-detection dataloader
/root/home/pvc/SECOND/test


2025-08-25 21:19:02.351588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756156742.368753   15128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756156742.373849   15128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756156742.387957   15128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756156742.387974   15128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756156742.387977   15128 computation_placer.cc:177] computation placer alr

[1, 1, 1]
['GROUP', {'num_groups': 8}]


25-08-25 21:19:04.727 - INFO: CD Model [cdmamba_modified] is created.
25-08-25 21:19:05.156 - INFO: CD Model moved to device: cuda
25-08-25 21:19:05.157 - INFO: Model parameters are on device: cuda:0
25-08-25 21:19:05.158 - INFO: ✓ Model successfully moved to GPU
25-08-25 21:19:05.158 - INFO: Number of classes for loss function: 6


In [4]:
#################
# Training loop #
#################
if opt['phase'] == 'train':
    best_mF1 = 0.0
    epoch_losses = []

    n_epochs = opt['train']['n_epoch']
    accumulation_steps = 2  # Effective batch size = batch_size * accumulation_steps

    for current_epoch in range(n_epochs):
        print("......Begin Training......")
        metric.clear()
        metric_seg.clear()
        cd_model.train()

        train_result_path = os.path.join(opt['path_cd']['result'], 'train', str(current_epoch))
        os.makedirs(train_result_path, exist_ok=True)

        # Log LR
        logger.info(f"lr: {optimizer.param_groups[0]['lr']:.7f}\n ")

        epoch_loss = 0.0

        # Prepare limited/iterable loader if max_train_batches set
        _max_train = getattr(args, 'max_train_batches', 0) or 0
        _train_total = min(len(train_loader), _max_train) if _max_train > 0 else len(train_loader)
        _train_iter = islice(train_loader, _max_train) if _max_train > 0 else train_loader

        # Zero grad at start of accumulation window
        optimizer.zero_grad(set_to_none=True)

        for current_step, batch in enumerate(tqdm(_train_iter, total=_train_total,
                                                 desc=f"Train {current_epoch}/{n_epochs}")):
            # ------------------ Fetch & move data ------------------
            train_im1 = batch['A'].to(device, non_blocking=True)
            train_im2 = batch['B'].to(device, non_blocking=True)

            # -------------- First-batch input debug (optional) --------------
            if current_step == 0:
                print("\n" + "="*60)
                print(f"EPOCH {current_epoch}, BATCH {current_step} - INPUT DEBUG INFO")
                print("="*60)
                print(f"Input T1 shape: {train_im1.shape}, dtype: {train_im1.dtype}")
                print(f"Input T1 range: [{train_im1.min():.4f}, {train_im1.max():.4f}]")
                print(f"Input T1 mean: {train_im1.mean():.4f}, std: {train_im1.std():.4f}")
                print(f"Input T2 shape: {train_im2.shape}, dtype: {train_im2.dtype}")
                print(f"Input T2 range: [{train_im2.min():.4f}, {train_im2.max():.4f}]")
                print(f"Input T2 mean: {train_im2.mean():.4f}, std: {train_im2.std():.4f}")
                print("-"*60)

            # ------------------ Forward (with AMP) ------------------
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = cd_model(train_im1, train_im2)

                # Unpack model outputs
                if isinstance(outputs, dict):
                    seg_logits_t1 = outputs.get('seg_t1', None)
                    seg_logits_t2 = outputs.get('seg_t2', None)
                    change_pred    = outputs.get('change', None)
                elif isinstance(outputs, (list, tuple)):
                    if len(outputs) == 3:
                        seg_logits_t1, seg_logits_t2, change_pred = outputs
                    else:
                        change_pred = outputs[0]
                        seg_logits_t1, seg_logits_t2 = None, None
                else:
                    change_pred = outputs
                    seg_logits_t1 = seg_logits_t2 = None

                # -------------- First-batch output debug (optional) --------------
                if current_step == 0:
                    print("\nOUTPUT DEBUG INFO:")
                    print("-"*60)
                    if seg_logits_t1 is not None:
                        print(f"Seg T1 logits - shape: {seg_logits_t1.shape}, range: [{seg_logits_t1.min():.4f}, {seg_logits_t1.max():.4f}]")
                        _pred = torch.argmax(seg_logits_t1, dim=1)
                        print(f"Seg T1 predictions - unique classes: {torch.unique(_pred).tolist()}")
                    if seg_logits_t2 is not None:
                        print(f"Seg T2 logits - shape: {seg_logits_t2.shape}, range: [{seg_logits_t2.min():.4f}, {seg_logits_t2.max():.4f}]")
                        _pred = torch.argmax(seg_logits_t2, dim=1)
                        print(f"Seg T2 predictions - unique classes: {torch.unique(_pred).tolist()}")
                    if change_pred is not None:
                        print(f"Change logits - shape: {change_pred.shape}, range: [{change_pred.min():.4f}, {change_pred.max():.4f}]")
                        _p = torch.softmax(change_pred, dim=1)
                        print(f"Change probs (class 0): [{_p[:,0].min():.4f}, {_p[:,0].max():.4f}]")
                        print(f"Change probs (class 1): [{_p[:,1].min():.4f}, {_p[:,1].max():.4f}]")
                    print("="*60 + "\n")

                # We no longer need raw inputs after forward
                del train_im1, train_im2

                # ------------------ Prepare labels ------------------
                seg_t1 = batch.get('L1', None)
                seg_t2 = batch.get('L2', None)
                change = batch.get('L', None)

                # First-batch GT debug
                if current_step == 0:
                    print("\nGROUND TRUTH DEBUG INFO:")
                    print("-"*60)
                    if seg_t1 is not None:
                        print(f"GT Seg T1 - shape: {seg_t1.shape}, dtype: {seg_t1.dtype}, uniq: {torch.unique(seg_t1).tolist()}")
                    else: print("GT Seg T1: None")
                    if seg_t2 is not None:
                        print(f"GT Seg T2 - shape: {seg_t2.shape}, dtype: {seg_t2.dtype}, uniq: {torch.unique(seg_t2).tolist()}")
                    else: print("GT Seg T2: None")
                    if change is not None:
                        print(f"GT Change - shape: {change.shape}, dtype: {change.dtype}, uniq: {torch.unique(change).tolist()}")
                        if change.numel() > 0:
                            _cr = (change == 1).float().mean().item()
                            print(f"GT Change pixel ratio: {_cr:.4f}")
                    else: print("GT Change: None")
                    print("-"*60)

                # Fallback for missing L1/L2
                if (seg_t1 is None) or (seg_t2 is None):
                    if change is not None:
                        seg_t1 = change
                        seg_t2 = change
                    else:
                        # create dummy zeros to match change_pred spatial dims
                        b, _, h, w = change_pred.shape
                        seg_t1 = torch.zeros((b, h, w), dtype=torch.long)
                        seg_t2 = torch.zeros((b, h, w), dtype=torch.long)

                # Ensure proper dtype/device
                if isinstance(seg_t1, torch.Tensor): seg_t1 = seg_t1.to(device).long()
                if isinstance(seg_t2, torch.Tensor): seg_t2 = seg_t2.to(device).long()
                if isinstance(change, torch.Tensor): change = change.to(device).long()

                # ------------------ Compute loss (NO thresholding) ------------------
                if opt['model']['loss'] == 'extended_triplet':
                    # Expect (seg_t1, seg_t2, change_pred)
                    assert isinstance(outputs, (tuple, list)) and len(outputs) == 3, \
                        "Expected model to return (seg_t1, seg_t2, change)"
                    seg_logits_t1, seg_logits_t2, change_pred = outputs

                    # TripletChangeSegLoss expects a 1-channel change logit
                    change_u = change_pred if change_pred.shape[1] == 1 else change_pred[:, 1:2]
                    change_bin = normalize_change_target(seg_t1, seg_t2, change)  # [B,H,W] long {0,1}

                    preds  = (seg_logits_t1, seg_logits_t2, change_u)
                    labels = {'seg_t1': seg_t1, 'seg_t2': seg_t2, 'change': change_bin}

                    if current_step == 0:
                        logger.info(f"[TRAIN dtype-check] change_bin: shape={tuple(change_bin.shape)}, dtype={change_bin.dtype}, device={change_bin.device}")
                        try:
                            _derived = normalize_change_target(seg_t1, seg_t2, None)
                            mism = (_derived != change_bin).float().mean().item()
                            logger.info(f"[TRAIN consistency] derived_vs_change_bin_mismatch={mism:.6f}")
                        except Exception as e:
                            logger.warning(f"[TRAIN consistency] compare failed: {e}")

                    raw_loss, ext_loss = loss_fun(preds, labels)  # scalar tensor
                    loss_dict = {
                        'seg_t1': ext_loss.get('seg_t1'),
                        'seg_t2': ext_loss.get('seg_t2'),
                        'change': ext_loss.get('cd')
                    }

                else:
                    # 2-class change head
                    if isinstance(outputs, (tuple, list)) and len(outputs) >= 3:
                        seg_logits_t1, seg_logits_t2, change_pred = outputs
                    else:
                        change_pred = outputs
                        # Create dummy seg logits for consistency in logging
                        b, _, h, w = change_pred.shape
                        seg_logits_t1 = torch.zeros((b, opt['model']['n_classes'], h, w),
                                                    device=change_pred.device, dtype=change_pred.dtype)
                        seg_logits_t2 = torch.zeros_like(seg_logits_t1)

                    change_bin = normalize_change_target(seg_t1, seg_t2, change)  # [B,H,W] long {0,1}

                    if current_step == 0:
                        logger.info(f"[TRAIN dtype-check] change_bin: shape={tuple(change_bin.shape)}, dtype={change_bin.dtype}, device={change_bin.device}")

                    raw_loss = loss_fun_change(change_pred, change_bin)
                    loss_dict = {'seg_t1': 0.0, 'seg_t2': 0.0, 'change': raw_loss.item()}

                # Scale for grad accumulation
                train_loss = raw_loss / accumulation_steps

            # ------------------ Backward & Step (AMP-aware) ------------------
            scaler.scale(train_loss).backward()

            do_step = ((current_step + 1) % accumulation_steps == 0) or ((current_step + 1) == _train_total)
            if do_step:
                torch.nn.utils.clip_grad_norm_(cd_model.parameters(), max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            # ------------------ Debug loss ------------------
            if current_step == 0:
                print("\nLOSS DEBUG INFO:")
                print("-"*60)
                print(f"Total train loss (scaled): {train_loss.item():.6f}")
                print(f"Loss components - seg_t1: {loss_dict['seg_t1']}, seg_t2: {loss_dict['seg_t2']}, change: {loss_dict['change']}")
                print(f"Loss requires_grad: {train_loss.requires_grad}")
                print("-"*60)

            # ------------------ Predictions for metrics/vis ------------------
            with torch.no_grad():
                pred_seg_t1 = torch.argmax(seg_logits_t1, dim=1)
                pred_seg_t2 = torch.argmax(seg_logits_t2, dim=1)
                change_p1   = torch.softmax(change_pred, dim=1)[:, 1, :, :]
                pred_change_bin = (change_p1 > args.change_threshold).long()

            # Log first-batch images (guarded)
            log_images_this_step = (current_step == 0) and WANDB_ENABLED

            if log_images_this_step:
                # Prepare GT visualizations
                seg_t1_np = seg_t1[0].detach().cpu().numpy()
                seg_t2_np = seg_t2[0].detach().cpu().numpy()
                gt_seg_t1_img = create_color_mask(seg_t1[0], num_classes=opt['model']['n_classes']) if seg_t1_np.ndim != 3 else seg_t1_np.astype(np.uint8)
                gt_seg_t2_img = create_color_mask(seg_t2[0], num_classes=opt['model']['n_classes']) if seg_t2_np.ndim != 3 else seg_t2_np.astype(np.uint8)
                gt_change_for_log = normalize_change_target(seg_t1, seg_t2, change)

                # Convert inputs from batch for logging
                A0 = batch['A'][0].detach().cpu()
                B0 = batch['B'][0].detach().cpu()
                def _norm_img(img):
                    if img.min() < 0:
                        img = (img + 1.0) / 2.0
                    img = (img * 255.0).clamp(0, 255).byte()
                    return img.permute(1,2,0).numpy() if img.ndim == 3 else img.numpy()

                # Prob heatmaps
                seg_t1_probs = torch.softmax(seg_logits_t1[0], dim=0)
                seg_t2_probs = torch.softmax(seg_logits_t2[0], dim=0)
                change_probs = torch.softmax(change_pred[0], dim=0)

                wandb.log({
                    "train/pred_seg_t1": [wandb.Image(create_color_mask(pred_seg_t1[0], num_classes=opt['model']['n_classes']))],
                    "train/pred_seg_t2": [wandb.Image(create_color_mask(pred_seg_t2[0], num_classes=opt['model']['n_classes']))],
                    "train/pred_change": [wandb.Image(create_color_mask(pred_change_bin[0], num_classes=2))],
                    "train/pred_change_prob": [wandb.Image(change_probs[1].detach().cpu().numpy())],
                    "train/gt_seg_t1": [wandb.Image(gt_seg_t1_img)],
                    "train/gt_seg_t2": [wandb.Image(gt_seg_t2_img)],
                    "train/gt_change": [wandb.Image(create_color_mask(gt_change_for_log[0], num_classes=2))],
                    "train/input_T1": [wandb.Image(_norm_img(A0))],
                    "train/input_T2": [wandb.Image(_norm_img(B0))],
                    "global_step": current_epoch * len(train_loader) + current_step
                })

            # ------------------ Metrics ------------------
            # Change (binary)
            gt_bin = (normalize_change_target(seg_t1, seg_t2, change) > 0).long().detach()
            pred_np = pred_change_bin.detach().cpu().numpy().astype(np.uint8)
            gt_np   = gt_bin.detach().cpu().numpy().astype(np.uint8)
            current_score = metric.update_cm(pr=pred_np, gt=gt_np)

            # Segmentation (multi-class)
            pred_seg_t1_np = pred_seg_t1.detach().cpu().numpy().astype(np.uint8)
            pred_seg_t2_np = pred_seg_t2.detach().cpu().numpy().astype(np.uint8)
            gt_seg_t1_np   = seg_t1.detach().cpu().numpy().astype(np.uint8)
            gt_seg_t2_np   = seg_t2.detach().cpu().numpy().astype(np.uint8)
            seg_score_t1 = metric_seg.update_cm(pr=pred_seg_t1_np, gt=gt_seg_t1_np)
            seg_score_t2 = metric_seg.update_cm(pr=pred_seg_t2_np, gt=gt_seg_t2_np)
            seg_score_avg = (seg_score_t1 + seg_score_t2) / 2.0

            # Log batch metrics
            log_dict = {
                'train_loss': train_loss.item(),
                'train_running_acc': current_score.item(),
                'train_running_seg_mf1': seg_score_avg.item()
            }
            wandb.log(log_dict)

            # Periodic console log with GPU mem
            if current_step % opt['train']['train_print_iter'] == 0:
                gpu_info = ""
                if torch.cuda.is_available():
                    mem_alloc = torch.cuda.memory_allocated() / 1024**3
                    mem_resv  = torch.cuda.memory_reserved() / 1024**3
                    gpu_info = f", GPU Memory: {mem_alloc:.2f}GB/{mem_resv:.2f}GB"
                logger.info('[Training CD]. epoch: [%d/%d]. Iter: [%d/%d], CD_loss: %.5f, change_mF1: %.5f, seg_mF1: %.5f%s\n' %
                            (current_epoch, n_epochs, current_step, _train_total,
                             train_loss.item(), current_score.item(), seg_score_avg.item(), gpu_info))

            # Accumulate epoch loss
            epoch_loss += train_loss.item()

            # Cleanup per-iter (let caching handle the rest)
            del outputs, seg_logits_t1, seg_logits_t2, change_pred
            del seg_t1, seg_t2, change
            del pred_seg_t1, pred_seg_t2, pred_change_bin, change_p1
            del gt_bin

        # ------------------ Epoch summary ------------------
        scores = metric.get_scores()          # change (binary)
        epoch_acc = scores['mf1']
        scores_seg = metric_seg.get_scores()  # segmentation (multi-class)
        epoch_seg_mf1  = scores_seg['mf1']
        epoch_seg_miou = scores_seg['miou']
        epoch_seg_acc  = scores_seg['acc']

        avg_epoch_loss = (epoch_loss / max(1, _train_total))
        epoch_losses.append(avg_epoch_loss)

        wandb.log({
            'train/epoch_mF1_change': epoch_acc,
            'train/epoch_mIoU_change': scores.get('miou', 0),
            'train/epoch_OA_change': scores.get('acc', 0),
            'train/epoch_mF1_seg': epoch_seg_mf1,
            'train/epoch_mIoU_seg': epoch_seg_miou,
            'train/epoch_OA_seg': epoch_seg_acc,
            'train/epoch_loss': avg_epoch_loss,
            'train_epoch_mf1': epoch_acc,   # backward-compat key
            'train_epoch_loss': avg_epoch_loss,
            'epoch': current_epoch
        })

        logger.info(f'Training - Epoch: {current_epoch}, Avg Loss: {avg_epoch_loss:.5f}, '
                    f'Change mF1: {epoch_acc:.5f}, Seg mF1: {epoch_seg_mf1:.5f}')
        if len(epoch_losses) > 1:
            trend = "↓" if epoch_losses[-1] < epoch_losses[-2] else "↑"
            logger.info(f'Loss trend: {trend} (Prev: {epoch_losses[-2]:.5f}, Curr: {epoch_losses[-1]:.5f})')


25-08-25 21:19:05.210 - INFO: lr: 0.0001000
 


......Begin Training......


Train 0/1:   0%|                                                                                                            | 0/1 [00:00<?, ?it/s]


EPOCH 0, BATCH 0 - INPUT DEBUG INFO
Input T1 shape: torch.Size([2, 3, 512, 512]), dtype: torch.float32
Input T1 range: [-0.6863, 1.0000]
Input T1 mean: 0.0461, std: 0.3000
Input T2 shape: torch.Size([2, 3, 512, 512]), dtype: torch.float32
Input T2 range: [-0.7569, 1.0000]
Input T2 mean: 0.2041, std: 0.3695
------------------------------------------------------------


25-08-25 21:19:07.481 - INFO: [TRAIN dtype-check] change_bin: shape=(2, 512, 512), dtype=torch.int64, device=cuda:0
25-08-25 21:19:07.491 - INFO: [TRAIN consistency] derived_vs_change_bin_mismatch=0.000000



OUTPUT DEBUG INFO:
------------------------------------------------------------
Seg T1 logits - shape: torch.Size([2, 6, 512, 512]), range: [-0.5020, 0.5610]
Seg T1 predictions - unique classes: [0, 1, 2, 3, 4, 5]
Seg T2 logits - shape: torch.Size([2, 6, 512, 512]), range: [-0.4036, 0.7056]
Seg T2 predictions - unique classes: [0, 2, 3, 4, 5]
Change logits - shape: torch.Size([2, 2, 512, 512]), range: [-0.4404, 0.2764]
Change probs (class 0): [0.3636, 0.5252]
Change probs (class 1): [0.4748, 0.6364]


GROUND TRUTH DEBUG INFO:
------------------------------------------------------------
GT Seg T1 - shape: torch.Size([2, 512, 512]), dtype: torch.int64, uniq: [0, 2, 3, 4, 5]
GT Seg T2 - shape: torch.Size([2, 512, 512]), dtype: torch.int64, uniq: [0, 2, 3, 4, 5]
GT Change - shape: torch.Size([2, 512, 512]), dtype: torch.int64, uniq: [0, 1]
GT Change pixel ratio: 0.1047
------------------------------------------------------------

LOSS DEBUG INFO:
------------------------------------------

25-08-25 21:19:10.142 - INFO: [Training CD]. epoch: [0/1]. Iter: [0/1], CD_loss: 2.11778, change_mF1: 0.09370, seg_mF1: 0.08631, GPU Memory: 0.20GB/23.61GB

Train 0/1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.76s/it]
25-08-25 21:19:10.179 - INFO: Training - Epoch: 0, Avg Loss: 2.11778, Change mF1: 0.09370, Seg mF1: 0.08803
